In [31]:
from dotenv import load_dotenv
import json, os, time, requests
import pprint

In [32]:
MODEL_NAME = "gemma3:27b"

load_dotenv()

#pegar a variável de sistema operacional ollama_endpoint   
OLLAMA_ENDPOINT = os.getenv("OLLAMA_ENDPOINT")

In [33]:
prompt_template = """
Quero que você gere um dataset sintético em português (PT-BR) no estilo cômico e impaciente do personagem "Pato Donald",
copiando falas oficiais da Disney quando estiverem disponíveis. O objetivo é criar exemplos para treinar um modelo que escreva como ele falaria:
- impaciente, resmungão, confuso às vezes;
- com humor leve e frases curtas;
- interjeições típicas como "Quá-quá!", "Grrr!", "Hmpf!" e "Bah!".

Gere apenas linhas JSON válidas, sem explicações, blocos de código ou comentários.
Cada linha deve ser um objeto JSON neste formato exato:

{{
  "id": "ex-**{{id}}**",
  "user": "<pergunta curta e natural em português>",
  "assistant": "<resposta curta no estilo 'Donald'>",
  "style_tags": ["<2–3 tags entre: IRRITADO, RESMUNGO, CÔMICO, CONFUSO, ANIMADO, SARCÁSTICO, RESILIENTE, ANEDÓTICO, AUTOIRONIA, TECNOLOGIA, SERENO, OTIMISTA_PRÁTICO>"],
  "constraints": {{"caps_max_ratio": 0.2, "min_len": 8, "max_len": 80}}
}}

Regras:
- Cada resposta deve incluir 1 a 2 interjeições ("Quá-quá!", "Grrr!", etc.).
- Evite texto TODO EM MAIÚSCULAS.
- Se estiver disponível pode usar falas de personagens oficiais.
- Varie os temas (rotina, frustração, tecnologia, comida/panquecas, trabalho, conselhos, humor).
- IDs devem seguir a numeração a partir de {start_id}.
- Nunca repita perguntas ou respostas.
Gere exatamente {count} linhas JSON, sem texto extra."""

In [35]:
# Configurações
OUTPUT_DIR ="donald_synthetic_data_leo"
os.makedirs(OUTPUT_DIR, exist_ok=True)
TOTAL_SAMPLES = 1000
BATCH_SIZE = 200
start_id = 1

In [36]:
def call_ollama(prompt):
    """Gera texto usando Ollama com suporte a stream desativado."""
    payload = {
        "model": "gemma3:27b",
        "prompt": prompt,
        "stream": False
    }

    headers = {"Content-Type": "application/json"}

    response = requests.post(
        OLLAMA_ENDPOINT+"/api/generate",
        headers=headers,
        data=json.dumps(payload),
        timeout=600
    )

    if response.status_code != 200:
        raise RuntimeError(f"Erro {response.status_code}: {response.text}")

    # Quando stream=False, Ollama retorna um único JSON com campo "response"
    data = response.json()
    return data.get("response", "").strip()


In [ ]:
# Geração de dados
for batch_idx in range(TOTAL_SAMPLES // BATCH_SIZE):
    start = start_id + batch_idx * BATCH_SIZE
    prompt = prompt_template.format(start_id=f"ex-{start:04d}", count=BATCH_SIZE)
    print(f"Gerando lote {batch_idx + 1} linhas {start}-{start + BATCH_SIZE - 1}...")

    try:
        text = call_ollama(prompt)

        file_path = os.path.join(OUTPUT_DIR, f"donald_data_{start}_{start + BATCH_SIZE - 1}.jsonl")
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(text + "\n")
        print(f"Lote salvo em {file_path}")

        time.sleep(5)  # Pausa para evitar limites de taxa
    except Exception as e:
        print(f"Erro ao gerar lote {batch_idx + 1}: {e}")

Gerando lote 1 linhas 1-200...
